<a href="https://colab.research.google.com/github/sravani93817-cpu/mini-project-for-data-science/blob/main/mini_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Create sample CSV if it doesn't exist
data = {
    'education': [8,6,9,7,8,5,9,6,8,7,9,6,7,8,9],
    'skills_score': [85,70,95,78,82,60,90,68,80,75,92,65,72,84,96],
    'aptitude': [90,65,92,80,85,55,88,70,82,72,95,60,68,86,93],
    'communication': [75,80,88,70,85,65,90,60,78,72,88,68,70,79,91],
    'certifications': [2,1,3,1,2,0,3,1,2,1,3,0,1,2,3],
    'internships': [1,0,2,1,2,0,2,0,1,1,2,0,0,1,2],
    'projects': [3,2,4,2,3,1,4,2,3,2,4,1,2,3,4],
    'experience': [2,1,3,2,2,0,3,1,2,1,3,1,1,2,3],
    'interview_score': [80,60,90,75,82,50,88,65,79,70,92,58,66,81,94],
    'hired': [1,0,1,0,1,0,1,0,1,0,1,0,0,1,1] # 1=Selected, 0=Rejected
}
df = pd.DataFrame(data)
df.to_csv('candidates.csv', index=False)
print("Sample CSV 'candidates.csv' created with 15 candidates\n")

# 2. Load data
df = pd.read_csv('candidates.csv')

# 3. Create Hiring Score - weighted sum
weights = {
    'education': 0.10, 'skills_score': 0.25, 'aptitude': 0.15,
    'communication': 0.15, 'certifications': 0.08, 'internships': 0.05,
    'projects': 0.07, 'experience': 0.05, 'interview_score': 0.10
}
df['hiring_score'] = sum(df[col] * w for col, w in weights.items())

# 4. ML: Predict selection
X = df.drop(['hired', 'hiring_score'], axis=1)
y = df['hired']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
print(f"Model Accuracy: {accuracy_score(y_test, y_pred)*100:.1f}%\n")

# 5. Rank candidates by predicted probability
df['pred_prob'] = clf.predict_proba(X_scaled)[:, 1]
df_ranked = df.sort_values('pred_prob', ascending=False).reset_index(drop=True)

# 6. Clustering: Excellent, Strong, Average, Needs Improvement
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df_ranked['cluster'] = kmeans.fit_predict(X_scaled)

cluster_map = {0: 'Needs Improvement', 1: 'Average', 2: 'Strong', 3: 'Excellent'}
df_ranked['category'] = df_ranked['cluster'].map(cluster_map)

# 7. Visualizations
plt.figure(figsize=(8,5))
sns.histplot(df_ranked['pred_prob'], bins=15, kde=True, color='teal')
plt.title('Distribution of Hiring Probability')
plt.xlabel('Probability of Selection')
plt.ylabel('Number of Candidates')
plt.tight_layout()
plt.savefig('hiring_prob_dist.png')

plt.figure(figsize=(7,4))
order = ['Excellent','Strong','Average','Needs Improvement']
sns.countplot(x='category', data=df_ranked, order=order, palette='viridis')
plt.title('Candidate Distribution by Category')
plt.xlabel('Category')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig('candidate_categories.png')

# 8. Feature importance
feat_imp = pd.DataFrame({'feature': X.columns, 'importance': clf.feature_importances_})
feat_imp = feat_imp.sort_values('importance', ascending=False)
plt.figure(figsize=(8,5))
sns.barplot(x='importance', y='feature', data=feat_imp, palette='mako')
plt.title('Key Factors Influencing Hiring Decision')
plt.tight_layout()
plt.savefig('feature_importance.png')

# 9. Save ranked results
df_ranked.to_csv('ranked_candidates.csv', index=False)

# 10. Output
print("=== TOP 5 CANDIDATES ===")
print(df_ranked[['education','skills_score','hiring_score','pred_prob','category']].head(5))
print("\n=== CATEGORY COUNTS ===")
print(df_ranked['category'].value_counts())
print("\nFiles generated: ranked_candidates.csv, hiring_prob_dist.png, candidate_categories.png, feature_importance.png")